In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import sys
sys.dont_write_bytecode = True
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ.setdefault(variable, '1')
NOTEBOOK_NAME = '02_02_posterior_diagnostics_and_export.ipynb'
NOTEBOOK_RELATIVE_PATH = '02_02_posterior_diagnostics_and_export.ipynb'
WORKING_DIRECTORY = Path.cwd().resolve()
NOTEBOOK_CANDIDATES = [Path(os.environ['A8_NOTEBOOK_PATH']).expanduser()] if 'A8_NOTEBOOK_PATH' in os.environ else [WORKING_DIRECTORY / NOTEBOOK_NAME, WORKING_DIRECTORY / NOTEBOOK_RELATIVE_PATH]
NOTEBOOK_PATH = next((p.resolve() for p in NOTEBOOK_CANDIDATES if p.is_file()), None)
if NOTEBOOK_PATH is None:
    raise FileNotFoundError('Set A8_NOTEBOOK_PATH to this notebook, or run with its containing directory as the working directory')
ANALYSIS_ROOT = NOTEBOOK_PATH.parent


In [ ]:
from pathlib import Path as u01_Path
from pathlib import Path as u01_Path
import json as u01_json, re as u01_re
import numpy as u01_np
import pandas as u01_pd


In [ ]:
def u01_ddm_fit_hddm(samples=15000, burn=5000, chains=4, seed=20261021, data=None, output_dir=None, model_check=False, ppc_samples=100, export_long=False):
    import hddm as hddm
    u01_ddm_validate_hddm_backend(hddm)
    if data is not None and (not model_check):
        raise ValueError('Explicit in-memory data are reserved for labelled model checks')
    d = u01_observed_hddm_data() if data is None else data.copy()
    d['subj_idx'] = d.subj_idx.astype(str)
    ids = set(u01_participants().subject_id)
    if set(d.subj_idx) != ids or set(d.task) != {'stroop', 'gonogo'}:
        raise ValueError('HDDM input cohort or task mismatch')
    if not d.response.isin([0, 1]).all() or not u01_np.isfinite(d.rt).all() or (d.rt <= 0).any():
        raise ValueError('Positive observed RT in seconds; response codes 0/1; missing RT encoded 999')
    if burn >= samples or chains < 2:
        raise ValueError('Multiple chains and samples > burn are required')
    out = u01_Path(output_dir) if output_dir is not None else u01_ddm_OUT
    (out / 'tables').mkdir(parents=True, exist_ok=True)
    fit_state = None
    if not model_check:
        import hashlib as hashlib
        fit_state = dict(status='running', samples=samples, burn=burn, chains=chains, ppc_samples=ppc_samples, export_long=export_long, model_code_sha256=u01_hddm_source_signature(), input_sha256={name: hashlib.sha256(u01_input_path(name).read_bytes()).hexdigest() for name in ['Participants.csv', 'Behavior_data.csv', 'HDDM_data.csv']})
        u01_atomic_record(out / 'fit_state.json', fit_state)
    long_path = out / 'tables/HDDM_posterior_draws.csv'
    wrote = False
    means = []
    diagrows = []
    for task in ['stroop', 'gonogo']:
        g = d[d.task.eq(task)].copy().reset_index(drop=True)
        expected = {'congruent', 'neutral', 'incongruent'} if task == 'stroop' else {'go', 'nogo'}
        if set(g.condition) != expected or set(g.subj_idx) != ids:
            raise ValueError('Missing task subjects or conditions')
        if not g.groupby('subj_idx').condition.apply(lambda x: set(x) == expected).all():
            raise ValueError('A subject lacks a model condition')
        if task == 'stroop':
            if not g.stimulus.isin([0, 1]).all() or g.rt.ge(998).any():
                raise ValueError('Stroop needs observed RT and actual color-boundary coding')
        elif (g.response.eq(0) != g.rt.eq(999)).any() or g.loc[g.response.eq(1), 'rt'].ge(998).any():
            raise ValueError('Go/No-Go lower boundary = nonresponse and RT 999; upper boundary = observed Go response')
        task_traces = []
        for chain in range(chains):
            u01_np.random.seed(seed + chain + (10000 if task == 'gonogo' else 0))
            kwargs = dict(depends_on={'v': 'condition'}, include=['a', 'v', 't', 'st'] + (['z'] if task == 'gonogo' else []), group_only_nodes=['st'], p_outlier=0.05)
            model = hddm.HDDMStimCoding(g, stim_col='stimulus', split_param='v', **kwargs) if task == 'stroop' else hddm.HDDM(g, **kwargs)
            model.find_starting_values()
            model.sample(samples, burn=burn, dbname=str(out / f'{task}_chain{chain}.db'), db='pickle')
            model.save(str(out / f'{task}_chain{chain}.model'))
            traces = model.get_traces()
            if traces.empty or not u01_np.isfinite(traces.to_numpy(float)).all():
                raise RuntimeError('Missing/nonfinite MCMC draws')
            traces.to_csv(out / f'{task}_chain{chain}_draws.csv', index=False)
            task_traces.append(traces)
            for node in traces:
                match = u01_re.match('^(a|v|t|z)_subj(?:\\(([^)]+)\\))?\\.(.+)$', str(node))
                if match:
                    (par, condition, sid) = match.groups()
                    parameter = task + '_' + par + ('_' + condition if par == 'v' else '')
                    if sid not in ids or parameter not in u01_PARAMS:
                        raise ValueError('Unmapped subject parameter: ' + str(node))
                else:
                    (sid, parameter) = ('__group__', task + '_group_' + str(node))
                if export_long:
                    block = u01_pd.DataFrame(dict(task=task, chain=chain, draw=u01_np.arange(len(traces)), subject_id=sid, parameter=parameter, value=traces[node].to_numpy()))
                    block.to_csv(long_path, index=False, mode='a' if wrote else 'w', header=not wrote)
                    wrote = True
            if ppc_samples:
                ppc = u01_ddm_posterior_predictive_samples(model, task, ppc_samples, seed + chain)
                ppc.to_csv(out / f'{task}_chain{chain}_PPC.csv')
            del model
        if any((list(tr.columns) != list(task_traces[0].columns) or len(tr) != len(task_traces[0]) for tr in task_traces)):
            raise ValueError('MCMC chain node or draw mismatch')
        for node in task_traces[0]:
            match = u01_re.match('^(a|v|t|z)_subj(?:\\(([^)]+)\\))?\\.(.+)$', str(node))
            values = u01_np.stack([tr[node].to_numpy() for tr in task_traces])
            if match:
                (par, condition, sid) = match.groups()
                parameter = task + '_' + par + ('_' + condition if par == 'v' else '')
                means.append(dict(subject_id=sid, parameter=parameter, value=values.mean()))
            if model_check:
                import arviz as az
                diagrows.append(dict(task=task, node=node, r_hat=float(az.rhat(values, method='rank')), ess_bulk=float(az.ess(values, method='bulk')), ess_tail=float(az.ess(values, method='tail'))))
    mean_table = u01_pd.DataFrame(means).pivot(index='subject_id', columns='parameter', values='value').reset_index()
    if set(u01_PARAMS) - set(mean_table) or len(mean_table) != len(ids):
        raise ValueError('Incomplete subject posterior means')
    mean_table['data_status'] = 'MODEL_CHECK_SIMULATION_REFITTED_NOT_EMPIRICAL' if model_check else 'FITTED_FROM_SUPPLIED_OBSERVATIONS'
    mean_table.to_csv(out / 'tables/HDDM_parameters.csv', index=False)
    if model_check:
        diagnostics = u01_pd.DataFrame(diagrows)
        diagnostics['diagnostics_pass'] = diagnostics.r_hat.lt(1.01) & diagnostics.ess_bulk.gt(400) & diagnostics.ess_tail.gt(400)
        diagnostics.to_csv(out / 'tables/Recovery_fit_diagnostics.csv', index=False)
        (out / 'MODEL_CHECK_ONLY.json').write_text(u01_json.dumps(dict(status='PARAMETER_RECOVERY_SIMULATION_REFIT_NOT_EMPIRICAL', seed=seed, samples=samples, burn=burn, chains=chains)))
    else:
        u01_provenance('hddm_fit', ['Participants.csv', 'Behavior_data.csv', 'HDDM_data.csv'], settings=dict(samples=samples, burn=burn, chains=chains, ppc_samples=ppc_samples, seed=seed, p_outlier=0.05))
        fit_state['status'] = 'complete'
        u01_atomic_record(out / 'fit_state.json', fit_state)
    return out


In [ ]:
def u01_ddm_validate_hddm_backend(hddm):
    import inspect as inspect
    from scipy import stats as stats
    if str(hddm.__version__) != '1.0.1':
        raise RuntimeError('Manuscript requires HDDM 1.0.1; install and validate the specified environment')
    source = inspect.getsource(hddm.likelihoods.generate_wfpt_stochastic_class)
    tree = __import__('ast').parse(source)
    fn = next((n for n in tree.body[0].body if isinstance(n, __import__('ast').FunctionDef) and n.name == 'wfpt_like'))
    namespace = dict(hddm=hddm, np=u01_np, stats=stats, wp=dict(err=0.0001, n_st=2, n_sz=2, use_adaptive=1, simps_err=0.001, w_outlier=0.1))
    module = __import__('ast').Module(body=[fn], type_ignores=[])
    exec(compile(module, 'installed_hddm_missing_rt_check', 'exec'), namespace)
    for v in [-1.0, 0.0, 1.0]:
        expected = 0.4 if v == 0 else u01_np.expm1(-2 * 1.2 * 0.4 * v) / u01_np.expm1(-2 * 1.2 * v)
        for boundary in [-1, 1]:
            val = namespace['wfpt_like'](u01_pd.DataFrame({'rt': [boundary * 999.0]}), v, 0, 1.2, 0.4, 0, 0.2, 0.05, p_outlier=0.05)
            prob = expected if boundary == 1 else 1 - expected
            if not u01_np.isclose(val, u01_np.log(prob), rtol=1e-06, atol=1e-08):
                raise RuntimeError('Installed HDDM does not implement the documented implicit-boundary missing-RT likelihood')
    return True


In [ ]:
def u01_ddm_recovery_sample(model, task, position):
    from kabuki.analyze import _parents_to_random_posterior_sample as _parents_to_random_posterior_sample
    traces = model.get_traces()
    if not 0 <= position < len(traces):
        raise ValueError('Posterior draw index outside fitted chain')
    truth = []
    for (name, value) in traces.iloc[position].items():
        match = u01_re.match('^(a|v|t|z)_subj(?:\\(([^)]+)\\))?\\.(.+)$', str(name))
        if match:
            (par, condition, sid) = match.groups()
            parameter = task + '_' + par + ('_' + condition if par == 'v' else '')
            truth.append(dict(subject_id=sid, parameter=parameter, generating_value=float(value)))
    generated = []
    for (name, record) in model.iter_observeds():
        node = record['node']
        _parents_to_random_posterior_sample(node, pos=position)
        sample = node.random(add_outliers=True)
        observed = model.data.loc[node.value.index].copy()
        if len(sample) != len(observed):
            raise ValueError('Recovery generator changed trial counts')
        signed_rt = sample['rt'].to_numpy(float)
        observed['response'] = (signed_rt > 0).astype(int)
        observed['rt'] = u01_np.abs(signed_rt)
        if task == 'gonogo':
            observed.loc[observed.response.eq(0), 'rt'] = 999.0
        observed['task'] = task
        generated.append(observed)
    data = u01_pd.concat(generated).sort_index().reset_index(drop=True)
    if len(data) != len(model.data):
        raise ValueError('Recovery generation did not cover all observed trials')
    if len(truth) != len(u01_participants()) * (5 if task == 'stroop' else 5):
        raise ValueError('Incomplete generating subject parameters')
    return (data, u01_pd.DataFrame(truth))


In [ ]:
def u01_ddm_posterior_predictive_samples(model, task, samples=100, seed=20261021):
    rng = u01_np.random.default_rng(seed)
    rows = []
    original = model.data.sort_index().reset_index(drop=True)
    for sample in range(samples):
        position = int(rng.integers(len(model.get_traces())))
        (generated, _) = u01_ddm_recovery_sample(model, task, position)
        if not generated[['subj_idx', 'condition', 'stimulus']].equals(original[['subj_idx', 'condition', 'stimulus']]):
            raise ValueError('Posterior prediction lost original trial alignment')
        block = original.copy()
        block['rt_sampled'] = u01_np.where(generated.response.eq(1), generated.rt, -generated.rt)
        block['response_sampled'] = generated.response.to_numpy()
        block['data_status'] = 'POSTERIOR_PREDICTIVE_SIMULATION_WITH_APPENDED_OBSERVATIONS'
        block['sample'] = sample
        block['posterior_draw'] = position
        rows.append(block)
    return u01_pd.concat(rows, ignore_index=True)


In [ ]:
from pathlib import Path as u02_Path
import sys as u02_sys
from IPython.display import display as u02_display, Image as u02_Image
from pathlib import Path as u02_Path
import json as u02_json, re as u02_re
import numpy as u02_np
import pandas as u02_pd


In [ ]:
def u02_ddm_posterior_blocks(group_only=False):
    state_path = u02_ddm_OUT / 'fit_state.json'
    if state_path.is_file() and __import__('os').environ.get('A8_HDDM_INPUT_MODE', 'auto') != 'external':
        u02_verify_fitted_artifact(u02_ddm_OUT / 'tables/HDDM_posterior_draws.csv')
        state = u02_json.loads(state_path.read_text())
        ids = set(u02_participants().subject_id)
        for task in ['stroop', 'gonogo']:
            traces = [u02_pd.read_csv(u02_ddm_OUT / f'{task}_chain{chain}_draws.csv') for chain in range(state['chains'])]
            if any((list(t.columns) != list(traces[0].columns) or len(t) != state['samples'] - state['burn'] for t in traces)):
                raise ValueError('Incomplete or inconsistent saved posterior chains')
            for node in traces[0]:
                match = u02_re.match('^(a|v|t|z)_subj(?:\\(([^)]+)\\))?\\.(.+)$', str(node))
                if match:
                    if group_only:
                        continue
                    (par, condition, sid) = match.groups()
                    param = task + '_' + par + ('_' + condition if par == 'v' else '')
                    if sid not in ids or param not in u02_PARAMS:
                        raise ValueError('Unmapped posterior parameter: ' + str(node))
                else:
                    (sid, param) = ('__group__', task + '_group_' + str(node))
                values = u02_np.stack([t[node].to_numpy(float) for t in traces])
                if not u02_np.isfinite(values).all():
                    raise ValueError('Nonfinite saved posterior samples')
                yield (task, sid, param, values)
    else:
        d = u02_table('HDDM_posterior_draws.csv', unique=['task', 'chain', 'draw', 'subject_id', 'parameter'])
        if group_only:
            d = d[d.subject_id.eq('__group__')]
        for ((task, sid, param), g) in d.groupby(['task', 'subject_id', 'parameter']):
            values = g.pivot(index='chain', columns='draw', values='value').sort_index().to_numpy(float)
            if not u02_np.isfinite(values).all():
                raise ValueError('Unbalanced or nonfinite external posterior samples')
            yield (task, sid, param, values)


In [ ]:
def u02_ddm_chain_summaries():
    import arviz as az
    summaries = []
    (u02_ddm_OUT / 'tables').mkdir(parents=True, exist_ok=True)
    (u02_ddm_OUT / 'image').mkdir(exist_ok=True)
    import matplotlib as matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    for (task, sid, param, a) in u02_ddm_posterior_blocks():
        if a.shape[0] < 2 or a.shape[1] < 4 or (not u02_np.isfinite(a).all()):
            raise ValueError(f'Unbalanced/short MCMC chains: {task}/{sid}/{param}')
        fixed = param == 'stroop_z' and u02_np.all(a == 0.5)
        diag = {'mean': a.mean(), 'sd': a.std(ddof=1), 'hdi_low': az.hdi(a.ravel(), hdi_prob=0.95)[0], 'hdi_high': az.hdi(a.ravel(), hdi_prob=0.95)[1]}
        diag.update({'r_hat': u02_np.nan, 'ess_bulk': u02_np.nan, 'ess_tail': u02_np.nan} if fixed else {'r_hat': float(az.rhat(a, method='rank')), 'ess_bulk': float(az.ess(a, method='bulk')), 'ess_tail': float(az.ess(a, method='tail'))})
        summaries.append(dict(task=task, subject_id=sid, parameter=param, fixed=fixed, **diag))
        if sid == '__group__' or sid == u02_participants().subject_id.iloc[0]:
            (fig, axes) = plt.subplots(1, 2, figsize=(7, 2.5))
            for (c, v) in enumerate(a):
                axes[0].plot(v, lw=0.9, alpha=0.85, color=['#0072B2', '#E69F00', '#009E73', '#CC79A7'][c % 4], label=f'Chain {c + 1}')
            axes[0].legend(ncol=4, fontsize=7, loc='upper center', bbox_to_anchor=(0.5, -0.22), frameon=False)
            axes[0].set_title(param, fontsize=8, fontweight='bold', pad=6)
            axes[0].set_xlabel('Sampling iterations', fontweight='bold')
            axes[0].set_ylabel('Value', fontweight='bold')
            axes[1].hist(a.ravel(), bins=50, density=True)
            fig.tight_layout()
            fig.savefig(u02_ddm_OUT / 'image' / f'{task}_{param}_actual_chains.png', dpi=600)
            plt.close(fig)
    result = u02_pd.DataFrame(summaries)
    result['diagnostics_pass'] = result.fixed | result.r_hat.lt(1.01) & result.ess_bulk.gt(400) & result.ess_tail.gt(400)
    result.to_csv(u02_ddm_OUT / 'tables/HDDM_sampling_diagnostics.csv', index=False)
    individual = result[result.subject_id.ne('__group__')].pivot(index='subject_id', columns='parameter', values='mean').reset_index()
    missing = set(u02_PARAMS) - set(individual)
    if missing:
        raise ValueError(f'Actual draws missing parameters: {missing}')
    individual.to_csv(u02_ddm_OUT / 'tables/HDDM_parameters.csv', index=False)
    if (u02_ddm_OUT / 'fit_state.json').is_file():
        u02_provenance('mcmc_summaries', ['Participants.csv', 'Behavior_data.csv', 'HDDM_data.csv'])
    else:
        u02_provenance('mcmc_summaries', ['HDDM_posterior_draws.csv'])
    return result


In [ ]:
def u02_ddm_group_contrasts(rope_bounds):
    import arviz as az
    import matplotlib.pyplot as plt
    expected = {'stroop_incongruent_minus_congruent', 'gonogo_abs_go_minus_abs_nogo'}
    if set(rope_bounds) != expected:
        raise ValueError('Provide both named ROPE intervals explicitly')
    group = {(task, param): values for (task, sid, param, values) in u02_ddm_posterior_blocks(group_only=True)}
    if not group:
        raise ValueError('Group posterior draws are required')
    rows = []
    for (label, task, left, right, absolute) in [('stroop_incongruent_minus_congruent', 'stroop', 'incongruent', 'congruent', False), ('gonogo_abs_go_minus_abs_nogo', 'gonogo', 'go', 'nogo', True)]:
        (lo, hi) = map(float, rope_bounds[label])
        if not u02_np.isfinite([lo, hi]).all() or not lo < 0 < hi:
            raise ValueError('ROPE must be an explicit finite interval containing zero')
        (a, b) = [group[task, task + '_group_v(' + cond + ')'].ravel() for cond in [left, right]]
        if a.shape != b.shape:
            raise ValueError('Unpaired group posterior draws')
        delta = u02_np.abs(a) - u02_np.abs(b) if absolute else a - b
        if not u02_np.isfinite(delta).all():
            raise ValueError('Unpaired group posterior draws')
        hdi = az.hdi(delta, hdi_prob=0.95)
        fig = reference_difference(delta, float(delta.mean()), hdi, (lo, hi), float(u02_np.mean((delta >= lo) & (delta <= hi))), 'Incongruent − congruent drift rate' if task == 'stroop' else '|Go drift rate| − |No-Go drift rate|', task)
        (u02_ddm_OUT / 'image').mkdir(parents=True, exist_ok=True)
        fig.savefig(u02_ddm_OUT / 'image' / (label + '_posterior_ROPE.png'), dpi=600)
        plt.close(fig)
        rows.append(dict(contrast=label, draws=len(delta), posterior_mean=delta.mean(), HDI95_low=hdi[0], HDI95_high=hdi[1], ROPE_low=lo, ROPE_high=hi, posterior_mass_in_ROPE=u02_np.mean((delta >= lo) & (delta <= hi)), HDI_fully_outside_ROPE=bool(hdi[1] < lo or hdi[0] > hi)))
    out = u02_pd.DataFrame(rows)
    (u02_ddm_OUT / 'tables').mkdir(parents=True, exist_ok=True)
    out.to_csv(u02_ddm_OUT / 'tables/Group_posterior_contrasts_ROPE.csv', index=False)
    return out


In [ ]:
from pathlib import Path as u15_Path
import os as u15_os, hashlib as u15_hashlib, json as u15_json
from importlib import metadata as u15_metadata
import platform as u15_platform, sys as u15_sys
import numpy as u15_np
import pandas as u15_pd
u15_ROOT = ANALYSIS_ROOT
u15_DATA = u15_Path(u15_os.environ.get('A8_DATA_DIR', NOTEBOOK_PATH.parent)).expanduser().resolve()
u15_RESULTS = u15_Path(u15_os.environ.get('A8_OUTPUT_DIR', u15_ROOT / 'results')).expanduser().resolve()
u15_EXPECTED_N = int(u15_os.environ.get('A8_EXPECTED_N', '106'))
u15_PARAMS = ['stroop_a', 'stroop_v_congruent', 'stroop_v_neutral', 'stroop_v_incongruent', 'stroop_t', 'gonogo_a', 'gonogo_v_go', 'gonogo_v_nogo', 'gonogo_t', 'gonogo_z']
u15_INPUT_FOLDERS = {'MRI_manifest.csv': '7.task_fmri', 'Participants.csv': '1.behavior', 'ROI_masks.csv': '7.task_fmri', 'HDDM_posterior_draws.csv': '2.hddm_fitting', 'HDDM_parameters.csv': '2.hddm_fitting', 'Behavior_data.csv': '1.behavior', 'HDDM_data.csv': '2.hddm_fitting'}
u15_INPUT_FOLDERS.update({'MRI_preprocessing.csv': '7.task_fmri', 'MultiEcho_runs.csv': '7.task_fmri'})
u15_INPUT_FILES = {name: name for name in u15_INPUT_FOLDERS}
u15_INPUT_COLUMNS = {'Participants.csv': ('subject_id', 'age', 'sex', 'education'), 'Behavior_data.csv': ('subject_id', 'task', 'run', 'trial_id', 'condition', 'rt_s', 'response', 'correct'), 'HDDM_data.csv': ('subj_idx', 'task', 'run', 'trial_id', 'condition', 'stimulus', 'response', 'rt'), 'MRI_manifest.csv': ('subject_id', 'task', 'run', 'bold', 'mask', 'confounds', 'metadata', 'events'), 'ROI_masks.csv': ('roi_id', 'path', 'source_doi', 'independent_of_sample'), 'HDDM_parameters.csv': ('subject_id', *u15_PARAMS), 'HDDM_posterior_draws.csv': ('task', 'chain', 'draw', 'subject_id', 'parameter', 'value')}


In [ ]:
def u15_input_path(name):
    filename = u15_INPUT_FILES.get(name, name)
    p = u15_DATA / filename
    if not p.is_file() and 'A8_DATA_DIR' not in u15_os.environ:
        p = u15_ROOT / u15_INPUT_FOLDERS.get(name, '7.task_fmri') / filename
    if name == 'MRI_manifest.csv' and (u15_os.environ.get('A8_MRI_INPUT_MODE') == 'generated' or not p.is_file()):
        p = u15_RESULTS / '7.task_fmri/tables/MRI_manifest.csv'
    fitted = (u15_RESULTS / '2.hddm_fitting/fit_state.json').is_file() and u15_os.environ.get('A8_HDDM_INPUT_MODE', 'auto') != 'external'
    if name in {'HDDM_parameters.csv', 'HDDM_posterior_draws.csv'} and (fitted or not p.is_file()):
        p = u15_RESULTS / '2.hddm_fitting/tables' / name
    if not p.is_file():
        raise FileNotFoundError(f'Required observed input missing: {p}. Supply the required real input file; no example values are substituted.')
    return p


In [ ]:
def u15_table(name, columns=(), unique=None):
    p = u15_input_path(name)
    d = u15_pd.read_csv(p, dtype={'subject_id': str, 'subj_idx': str, 'run': str, 'trial_id': str})
    if d.empty:
        raise ValueError(f'{p.name}: no observed records; a header-only template is not data')
    missing = (set(columns) | set(u15_INPUT_COLUMNS.get(name, ()))) - set(d)
    if missing:
        raise ValueError(f'{p.name}: missing columns {sorted(missing)}')
    if unique and (d[list(unique)].isna().any().any() or d.duplicated(list(unique)).any()):
        raise ValueError(f'{p.name}: missing/duplicate key {unique}')
    for c in ('subject_id', 'subj_idx', 'trial_id'):
        if c in d and (d[c].isna().any() or d[c].str.strip().eq('').any() or d[c].ne(d[c].str.strip()).any()):
            raise ValueError(f'{p.name}: blank or whitespace-padded identifier in {c}')
    for c in ('data_status', 'source_status'):
        if c in d and d[c].astype(str).str.contains('synthetic|simulation|not empirical', case=False, regex=True).any():
            raise ValueError(f'{p.name}: simulated rows cannot enter this workflow')
    return d


In [ ]:
def u15_participants():
    d = u15_table('Participants.csv', ['subject_id', 'age', 'sex', 'education'], ['subject_id'])
    if len(d) != u15_EXPECTED_N:
        raise ValueError(f'Expected {u15_EXPECTED_N} participants, found {len(d)}; configure A8_EXPECTED_N deliberately.')
    for c in ['age', 'sex', 'education']:
        d[c] = u15_pd.to_numeric(d[c], errors='raise')
        if not u15_np.isfinite(d[c]).all():
            raise ValueError(f'Nonfinite {c}')
    if not d.sex.isin([0, 1]).all():
        raise ValueError('sex must use the documented 0/1 coding')
    return d.sort_values('subject_id').reset_index(drop=True)


In [ ]:
def u15_match_cohort(d, name, keys=('subject_id',), complete=True):
    ids = set(u15_participants().subject_id)
    if d.subject_id.isna().any() or not set(d.subject_id) <= ids:
        raise ValueError(f'{name}: unexpected or missing subject_id')
    if complete and set(d.subject_id) != ids:
        raise ValueError(f'{name}: cohort mismatch; do not fill missing participants')
    if d.duplicated(list(keys)).any():
        raise ValueError(f'{name}: duplicate {keys}')
    return d


In [ ]:
def u15_behavior():
    d = u15_table('Behavior_data.csv', ['subject_id', 'task', 'run', 'trial_id', 'condition', 'rt_s', 'response', 'correct'], ['subject_id', 'task', 'run', 'trial_id'])
    u15_match_cohort(d, 'Behavior_data', ('subject_id', 'task', 'run', 'trial_id'))
    if not d.task.isin(['stroop', 'gonogo']).all():
        raise ValueError('task must be stroop/gonogo')
    d['rt_s'] = u15_pd.to_numeric(d.rt_s, errors='raise')
    d['response'] = u15_pd.to_numeric(d.response, errors='raise')
    d['correct'] = u15_pd.to_numeric(d.correct, errors='raise')
    if not d.correct.isin([0, 1]).all() or not d.response.dropna().isin([0, 1]).all():
        raise ValueError('correct and observed response must be binary')
    if not u15_np.isfinite(d.rt_s.dropna()).all() or (d.rt_s.dropna() <= 0).any():
        raise ValueError('RTs must be in seconds; nonresponses have NaN RT')
    for (task, conds) in [('stroop', {'congruent', 'neutral', 'incongruent'}), ('gonogo', {'go', 'nogo'})]:
        g = d[d.task.eq(task)]
        if set(g.subject_id) != set(u15_participants().subject_id) or set(g.condition) != conds:
            raise ValueError(f'{task}: incomplete subjects/conditions')
        if not g.groupby('subject_id').condition.apply(lambda x: set(x) == conds).all():
            raise ValueError(f'{task}: a participant is missing a condition')
    return d


In [ ]:
def u15_source_signature():
    document = u15_json.loads(NOTEBOOK_PATH.read_text())
    source = '\n'.join((''.join(cell['source']) for cell in document['cells'] if cell['cell_type'] == 'code'))
    return u15_hashlib.sha256(source.encode()).hexdigest()


In [ ]:
def u15_runtime_versions():
    versions = {'python': u15_sys.version, 'platform': u15_platform.platform()}
    for name in ['numpy', 'scipy', 'pandas', 'statsmodels', 'scikit-learn', 'nilearn', 'nibabel', 'arviz', 'hddm', 'kabuki', 'pymc', 'matplotlib', 'openpyxl', 'antspyx', 'templateflow']:
        try:
            versions[name] = u15_metadata.version(name)
        except u15_metadata.PackageNotFoundError:
            versions[name] = None
    return versions


In [ ]:
def u15_atomic_record(path, record):
    path = u15_Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.pending')
    temp.write_text(u15_json.dumps(record, ensure_ascii=False, indent=2, allow_nan=False))
    temp.replace(path)


In [ ]:
def u15_provenance(stage, names, settings=None):
    records = []
    for name in names:
        path = u15_input_path(name)
        records.append({'file': str(path), 'sha256': u15_hashlib.sha256(path.read_bytes()).hexdigest()})
    record = dict(stage=stage, input_files=records, code_sha256=u15_source_signature(), runtime=u15_runtime_versions(), settings=settings or {})
    u15_atomic_record(u15_RESULTS / (stage + '_inputs.json'), record)
    return record


In [ ]:
def u15_verify_fitted_artifact(path):
    path = u15_Path(path).resolve()
    generated = (u15_RESULTS / '2.hddm_fitting/tables').resolve()
    if path.parent != generated:
        return
    state_path = generated.parent / 'fit_state.json'
    if not state_path.is_file():
        raise RuntimeError('Generated HDDM tables lack a completed fit record; rerun the fit or use explicitly supplied external estimates')
    state = u15_json.loads(state_path.read_text())
    if state.get('model_code_sha256') != u15_hddm_source_signature():
        raise RuntimeError('HDDM model implementation changed since fitting; refit before using generated estimates')
    if state.get('status') != 'complete':
        raise RuntimeError('HDDM fit is incomplete; downstream analyses cannot read its tables')
    for (name, expected) in state['input_sha256'].items():
        if u15_hashlib.sha256(u15_input_path(name).read_bytes()).hexdigest() != expected:
            raise RuntimeError('Observed inputs changed since HDDM fitting: ' + name)
    diagnostics = generated / 'HDDM_sampling_diagnostics.csv'
    if path.name == 'HDDM_parameters.csv':
        if not diagnostics.is_file() or diagnostics.stat().st_mtime_ns < state_path.stat().st_mtime_ns:
            raise RuntimeError('Run chain diagnostics for the completed fit before associations')
        check = u15_pd.read_csv(diagnostics, dtype={'subject_id': str})
        if check.empty or not check.diagnostics_pass.astype(str).str.lower().eq('true').all():
            raise RuntimeError('HDDM sampling diagnostics did not all pass')


In [ ]:
def u15_observed_hddm_data():
    d = u15_table('HDDM_data.csv', unique=['subj_idx', 'task', 'run', 'trial_id']).rename(columns={'subj_idx': 'subject_id'})
    raw = u15_behavior()
    keep = raw.task.eq('gonogo') | raw.task.eq('stroop') & raw.rt_s.notna() & raw.response.notna()
    raw = raw.loc[keep]
    keys = ['subject_id', 'task', 'run', 'trial_id']
    if set(map(tuple, d[keys].to_numpy())) != set(map(tuple, raw[keys].to_numpy())):
        raise ValueError('HDDM and behavioral trial sets differ; retain observed Stroop responses and every Go/No-Go trial')
    joined = d.merge(raw[keys + ['condition', 'rt_s', 'response', 'correct']], on=keys, suffixes=('', '_behavior'), validate='one_to_one')
    if not joined.condition.eq(joined.condition_behavior).all():
        raise ValueError('HDDM and behavioral condition coding differs')
    if not joined.response.isin([0, 1]).all() or not u15_np.isfinite(joined.rt).all():
        raise ValueError('Invalid HDDM response or RT')
    st = joined.task.eq('stroop')
    if not joined.loc[st, 'stimulus'].isin([0, 1]).all() or not joined.loc[st, 'response'].eq(joined.loc[st, 'response_behavior']).all():
        raise ValueError('Stroop stimulus/response coding mismatch')
    if not joined.loc[st, 'correct'].eq(joined.loc[st, 'response'].eq(joined.loc[st, 'stimulus']).astype(int)).all():
        raise ValueError('Stroop correctness differs from the color-boundary coding')
    ng = ~st
    observed = joined.response.eq(1) | st
    if not u15_np.allclose(joined.loc[observed, 'rt'], joined.loc[observed, 'rt_s'], rtol=1e-08, atol=1e-09):
        raise ValueError('HDDM RTs differ from observed behavioral RTs')
    if not joined.loc[ng, 'response'].eq(joined.loc[ng, 'response_behavior']).all():
        raise ValueError('Go/No-Go response coding mismatch')
    if not joined.loc[ng & ~observed, 'rt'].eq(999).all() or joined.loc[ng & ~observed, 'rt_s'].notna().any():
        raise ValueError('Go/No-Go nonresponses must have blank behavioral RT and HDDM RT 999')
    return d.rename(columns={'subject_id': 'subj_idx'})


In [ ]:
def u15_hddm_source_signature():
    import types as types
    fit = u01_ddm_fit_hddm
    backend = u01_ddm_validate_hddm_backend
    sample = u01_ddm_recovery_sample
    predictive = u01_ddm_posterior_predictive_samples
    import re as re

    def normalized(code):
        return [code.co_code.hex(), [normalized(x) if isinstance(x, types.CodeType) else repr(x) for x in code.co_consts], [re.sub('^u[0-9]+_', '', x) for x in code.co_names], list(code.co_varnames), code.co_argcount, code.co_kwonlyargcount]
    payload = [normalized(f.__code__) for f in [fit, backend, sample, predictive, u15_observed_hddm_data]]
    return u15_hashlib.sha256(u15_json.dumps(payload, sort_keys=True).encode()).hexdigest()


In [ ]:
u01_RESULTS = u15_RESULTS
u01_hddm_source_signature = u15_hddm_source_signature
u01_RESULTS = u15_RESULTS
u01_participants = u15_participants
u01_input_path = u15_input_path
u01_provenance = u15_provenance
u01_PARAMS = u15_PARAMS
u01_observed_hddm_data = u15_observed_hddm_data
u01_atomic_record = u15_atomic_record
u01_ddm_OUT = u01_RESULTS / '2.hddm_fitting'
u02_ROOT = u15_ROOT
u02_DATA = u15_DATA
u02_RESULTS = u15_RESULTS
u02_hddm_source_signature = u15_hddm_source_signature
u02_ROOT = u15_ROOT
u02_RESULTS = u15_RESULTS
u02_behavior = u15_behavior
u02_participants = u15_participants
u02_table = u15_table
u02_input_path = u15_input_path
u02_provenance = u15_provenance
u02_PARAMS = u15_PARAMS
u02_observed_hddm_data = u15_observed_hddm_data
u02_atomic_record = u15_atomic_record
u02_verify_fitted_artifact = u15_verify_fitted_artifact
u02_ddm_OUT = u02_RESULTS / '2.hddm_fitting'


In [ ]:
def a8_display_three_line(value, *args, **kwargs):
    import pandas as pd
    from IPython.display import display, HTML
    if isinstance(value, pd.DataFrame):
        html = value.to_html(index=False, border=0, escape=True)
        html = html.replace('<table ', '<table style="border-collapse:collapse;border-top:2px solid black;border-bottom:2px solid black;" ', 1)
        html = html.replace('<th>', '<th style="border:0;border-bottom:1px solid black;padding:7px 16px;text-align:center;">')
        html = html.replace('<td>', '<td style="border:0;padding:7px 16px;text-align:center;">')
        return display(HTML(html), *args, **kwargs)
    return display(value, *args, **kwargs)
u02_display = a8_display_three_line

def segment_intersects_rect(segment, rect):
    ((x0, y0), (x1, y1)) = segment
    (dx, dy) = (x1 - x0, y1 - y0)
    (lower, upper) = (0.0, 1.0)
    for (denominator, numerator) in zip((-dx, dx, -dy, dy), (x0 - rect[0], rect[2] - x0, y0 - rect[1], rect[3] - y0)):
        if abs(denominator) < 1e-12:
            if numerator < 0:
                return False
            continue
        ratio = numerator / denominator
        if denominator < 0:
            lower = max(lower, ratio)
        else:
            upper = min(upper, ratio)
        if lower > upper:
            return False
    return True

def finish_axis(ax):
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.spines['bottom'].set_linewidth(1.1)
    ax.set_yticks([])
    ax.tick_params(axis='x', labelsize=8, length=3)
    ax.grid(False)

def reference_difference(draws, mean, interval, rope, rope_fraction, xlabel, headline='', interval_label='95% HDI', zoom=False):
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.stats import gaussian_kde
    from matplotlib.lines import Line2D
    BLUE = '#009E73'
    ORANGE = '#CC79A7'
    (h0, h1) = interval
    rope_low, rope_high = (-float(rope), float(rope)) if np.isscalar(rope) else tuple(map(float, rope))
    (lo, hi) = np.quantile(draws, [0.0005, 0.9995])
    lo = min(lo, rope_low * 1.35)
    hi = max(hi, rope_high * 1.35)
    pad = (hi - lo) * 0.12
    lo -= pad
    hi += pad
    if (mean - lo) / (hi - lo) > 0.68:
        hi = max(hi, mean + (mean - lo) * 0.55)
    grid = np.linspace(lo, hi, 2200)
    kde = gaussian_kde(draws)
    density = kde(grid)
    peak = float(density.max())
    (fig, ax) = plt.subplots(figsize=(3.4, 2.4))
    ax.axvspan(rope_low, rope_high, facecolor='#BDBDBD', edgecolor='none', linewidth=0, alpha=0.65, zorder=0)
    ax.fill_between(grid, density, color=BLUE, alpha=0.7, linewidth=0, zorder=2)
    ax.plot(grid, density, color=BLUE, linewidth=1.1, zorder=3)
    ax.hlines(peak * 0.025, rope_low, rope_high, color=ORANGE, linewidth=1.6, zorder=4)
    ax.hlines(peak * 0.017, rope_low, rope_high, color='#777777', linewidth=3, alpha=0.6, zorder=3)
    ax.text(rope_low, peak * 0.235, f'{rope_low:.3f}', color=ORANGE, fontweight='bold', fontsize=8.5, ha='right', va='center')
    ax.text(rope_high, peak * 0.235, f'{rope_high:.3f}', color=ORANGE, fontweight='bold', fontsize=8.5, ha='left', va='center')
    ax.hlines(peak * 0.022, h0, h1, color='black', linewidth=1.8, zorder=5)
    offset = 0.0
    ax.text(h0 - offset, peak * 0.045, f'{h0:.3f}', fontsize=8, ha='right', va='bottom', zorder=6)
    ax.text(h1 + offset, peak * 0.045, f'{h1:.3f}', fontsize=8, ha='left', va='bottom', zorder=6)
    ax.text(mean, peak * 0.88, f'mean={mean:.3f}', fontsize=9, ha='center', va='bottom', zorder=6)
    ax.text(0.54, 0.5, f'{100 * rope_fraction:.2f}% in ROPE', transform=ax.transAxes, color=ORANGE, fontweight='bold', fontsize=9, ha='center', va='center', zorder=6)
    ax.text(0.54, 0.35, '95% HDI', transform=ax.transAxes, fontsize=9, ha='center', va='center', zorder=6)
    ax.legend([Line2D([0], [0], color=BLUE, lw=3), Line2D([0], [0], color=ORANGE, marker='s', markersize=6, linestyle='None')], ['95% HDI', 'ROPE'], loc='upper right', fontsize=7.5, frameon=False, handlelength=1.5)
    ax.set_ylim(-peak * 0.05, peak * 1.2)
    ax.set_xlim(lo, hi)
    ax.set_xticks(np.linspace(lo, hi, 5))
    from matplotlib.ticker import FormatStrFormatter
    ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    ax.set_xlabel(xlabel, fontsize=9, labelpad=5)
    ax.set_ylabel('Density', fontsize=9, labelpad=9)
    if headline:
        ax.set_title(headline, fontsize=9.5, pad=8)
    finish_axis(ax)
    ax.spines['bottom'].set_linewidth(1.8)
    ax.tick_params(axis='x', length=0)
    fig.subplots_adjust(left=0.12, right=0.985, bottom=0.2, top=0.89)
    from matplotlib.transforms import Bbox
    curve = np.column_stack([grid, density])
    curve_pixels = ax.transData.transform(curve)

    def crosses(rect):
        bb = (rect.x0, rect.y0, rect.x1, rect.y1)
        for (q, r) in zip(curve_pixels[:-1], curve_pixels[1:]):
            if max(q[0], r[0]) < rect.x0 or min(q[0], r[0]) > rect.x1:
                continue
            if segment_intersects_rect((tuple(q), tuple(r)), bb):
                return True
        return False
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    legend_box = ax.get_legend().get_window_extent(renderer).expanded(1.03, 1.08)
    placed = []
    texts = sorted(ax.texts, key=lambda t: 0 if t.get_text().startswith('mean=') else 2 if 'in ROPE' in t.get_text() or t.get_text() == '95% HDI' else 1)
    for text in texts:
        point = ax.transAxes.inverted().transform(text.get_transform().transform(text.get_position()))
        (x0, y0) = point
        ha0 = text.get_ha()
        text.set_transform(ax.transAxes)
        label = text.get_text()
        if label.startswith('mean='):
            candidates = [(x, y, 'center') for y in [y0, 0.91, 0.8, 0.72] for x in [x0, x0 - 0.1, x0 + 0.1, x0 - 0.18, x0 + 0.18]]
        elif 'in ROPE' in label or label == '95% HDI':
            candidates = [(x, y, 'center') for x in [x0, 0.48, 0.62, 0.7, 0.3, 0.2, 0.82] for y in [y0, y0 - 0.06, y0 + 0.06, y0 - 0.12, y0 + 0.12]]
        else:
            candidates = [(x0, y, ha) for y in [y0, y0 + 0.06, y0 + 0.12, y0 + 0.18, y0 + 0.24, y0 + 0.3, y0 + 0.36] for ha in [ha0, 'left' if ha0 == 'right' else 'right']]
        found = False
        for (x, y, ha) in candidates:
            text.set_position((x, y))
            text.set_ha(ha)
            rect = text.get_window_extent(renderer).padded(2.0)
            if rect.x0 < ax.bbox.x0 + 1 or rect.x1 > ax.bbox.x1 - 1 or rect.y0 < ax.bbox.y0 + 3 or (rect.y1 > ax.bbox.y1 - 1):
                continue
            if rect.overlaps(legend_box) or any((rect.overlaps(b) for b in placed)) or crosses(rect):
                continue
            placed.append(rect)
            found = True
            break
        if not found:
            raise RuntimeError('Unable to place screenshot annotation without overlap: ' + label)
    fig._export_pad_inches = 0.07
    return fig


In [ ]:
u02_parameter_summary = u02_ddm_chain_summaries()
u02_display(u02_parameter_summary)


In [ ]:
import pandas as u02_pd
u02_parameter_file = u02_ddm_OUT / 'tables/HDDM_parameters.csv'
u02_parameter_table = u02_pd.read_csv(u02_parameter_file, dtype={'subject_id': str})
assert u02_parameter_table.subject_id.notna().all() and u02_parameter_table.subject_id.is_unique
u02_export_path = u02_RESULTS / '2.hddm_fitting/tables/HDDM_parameters.xlsx'
u02_export_path.parent.mkdir(parents=True, exist_ok=True)
u02_parameter_table.to_excel(u02_export_path, index=False, sheet_name='Fitted_parameters')
u02_display(u02_parameter_table)


In [ ]:
u02_rope_bounds = {}
for u02_contrast in ['stroop_incongruent_minus_congruent', 'gonogo_abs_go_minus_abs_nogo']:
    u02_values = input(u02_contrast + ' ROPE lower,upper (required): ').split(',')
    if len(u02_values) != 2:
        raise ValueError('Two explicit ROPE endpoints are required')
    u02_rope_bounds[u02_contrast] = tuple((float(x) for x in u02_values))
u02_display(u02_ddm_group_contrasts(u02_rope_bounds))
